# PMF를 만들고 dropout-like mask를 관측하기

이번 실습은 이산 확률분포를 코드로 옮기는 가장 작은 단계입니다. 먼저 네 개 활성값에서 유지 개수가 나오는 분포를 만들고, 이어서 같은 가정으로 여러 마스크를 뽑아 이론값과 관측값을 비교합니다.

각 실습은 구현 → fixture → `check_e##()` 순서로 실행하세요.

In [1]:
import math
import numpy as np


## 네 활성값의 유지 개수 PMF 만들기

### 목적
네 활성값이 각각 같은 유지 확률을 가지며 서로 독립일 때, 유지 개수별 확률을 배열로 만듭니다.

### 요구 사항
`binomial_pmf`는 정수 시행 수와 유지 확률을 받아 가능한 유지 개수 배열과 같은 길이의 확률 배열을 반환해야 합니다. 확률 배열의 각 원소는 해당 유지 개수의 확률질량이며, 모든 원소는 음수가 아니고 합은 1이어야 합니다. 시행 수가 음수이거나 유지 확률이 0과 1 사이가 아니면 `ValueError`를 발생시켜야 합니다.

### 작은 사례
시행 수가 1이면 가능한 유지 개수는 0과 1 두 개입니다. 유지 확률이 0이면 항상 0개, 유지 확률이 1이면 항상 1개가 나옵니다.

<details><summary>힌트 1</summary>가능한 유지 개수마다 조합 수가 서로 다릅니다. `math.comb`가 그 조합 수를 계산합니다.</details>

<details><summary>힌트 2</summary>확률 배열을 만든 뒤에는 `np.sum`으로 전체 질량을 확인할 수 있습니다.</details>

In [2]:
from math import comb


def binomial_pmf(trials: int, keep_prob: float) -> tuple[np.ndarray, np.ndarray]:
    """Return retained-count support and one probability mass per support value."""
    if trials < 0:
        raise ValueError("trials must be nonnegative")
    if not 0.0 <= keep_prob <= 1.0:
        raise ValueError("keep_prob must be between 0 and 1")

    support = np.arange(trials + 1, dtype=int)
    # Local contract: `support` contains every retained count from zero through `trials`.
    # Return a float `probs` of the same shape, with the binomial mass for each support value.
    # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
    # TODO: 각 유지 개수에 대한 조합 수와 유지/제거 확률을 사용해 probs를 만드세요.
    probs =  [(keep_prob ** i) * comb(trials, i) * ((1 - keep_prob) ** (trials - i)) for i, sup in enumerate(support)]
    probs = np.asarray(probs, dtype=float)
    # 학습자 편집 구간 끝
    return support, probs

# 실행 순서: 이 셀 → 바로 아래 fixture → check_e01()

In [3]:
support, probs = binomial_pmf(trials=4, keep_prob=0.75)
print("support:", support)
print("probs:", probs)
print("total mass:", np.sum(probs))

support: [0 1 2 3 4]
probs: [0.00390625 0.046875   0.2109375  0.421875   0.31640625]
total mass: 1.0


In [4]:
def check_e01() -> None:
    support, probs = binomial_pmf(4, 0.75)
    np.testing.assert_array_equal(support, np.array([0, 1, 2, 3, 4]))
    np.testing.assert_allclose(probs, np.array([1, 12, 54, 108, 81], dtype=float) / 256)
    np.testing.assert_allclose(np.sum(probs), 1.0)
    np.testing.assert_allclose(binomial_pmf(1, 0.0)[1], np.array([1.0, 0.0]))
    np.testing.assert_allclose(binomial_pmf(1, 1.0)[1], np.array([0.0, 1.0]))
    for args in ((-1, 0.5), (4, -0.1), (4, 1.1)):
        try:
            binomial_pmf(*args)
        except ValueError:
            pass
        else:
            raise AssertionError(f"ValueError expected for {args}")
    print("check_e01 passed")

check_e01()

check_e01 passed


### 결과 해석

<!-- TODO: 네 개 활성값의 유지 개수가 왜 여러 값이 될 수 있는지, 그리고 확률 배열의 합이 1이어야 하는 이유를 2~4문장으로 설명하세요. -->

네 활성값은 각각 유지 또는 제거될 수 있으므로 유지 개수는 0개부터 4개까지 나올 수 있다. 각 유지 개수의 확률을 모두 더하면 어떤 결과 하나는 반드시 발생하므로 전체 확률은 1이어야 한다. 유지 확률이 0.75일 때는 3개가 유지될 가능성이 가장 크지만, 다른 개수도 나올 수 있다.

## 여러 dropout-like mask를 뽑아 이론과 비교하기

### 목적
동일한 네 활성값 마스크를 여러 번 생성해, 실제 유지 개수들의 평균과 분산을 관측합니다.

### 요구 사항
`sample_retained_counts`는 양의 마스크 수, 양의 활성값 수, 0과 1 사이의 유지 확률, 정수 시드를 받습니다. 시드가 같으면 같은 정수 유지 개수 배열을 반환해야 하며, 배열 길이는 마스크 수와 같아야 합니다. 반환한 딕셔너리에는 관측 유지 개수, 관측 평균, 관측 분산, 이론 평균, 이론 분산을 담아야 합니다. 관측 분산은 마스크 수로 나누는 모집단 분산 규칙인 `ddof=0`을 사용합니다. 잘못된 개수나 유지 확률은 `ValueError`를 발생시켜야 합니다. 구조화된 dropout은 한 채널이나 묶음이 함께 유지되거나 제거될 수 있는 방식입니다. 관측 평균·분산이 이론값과 달라도 오류라고 단정할 수 없는 이유와, 구조화된 dropout에서 이 실습의 독립 가정이 왜 깨질 수 있는지를 3~5문장으로 설명하세요.

### 작은 사례
활성값이 하나이고 유지 확률이 0이면 모든 관측 유지 개수는 0입니다. 유지 확률이 1이면 모든 관측 유지 개수는 1입니다.

<details><summary>힌트 1</summary>이 문제의 가정은 고정된 활성값 수, 공통 유지 확률, 독립적인 유지 여부입니다. NumPy의 난수 생성기는 시드로 고정하세요.</details>

<details><summary>힌트 2</summary>관측 평균과 관측 분산은 `counts`에서, 이론값은 활성값 수와 유지 확률에서 계산합니다.</details>

In [5]:
def sample_retained_counts(
    n_masks: int, n_activations: int, keep_prob: float, seed: int
) -> dict[str, np.ndarray | float]:
    """Sample retained counts and summarize the observed and theoretical distribution."""
    if n_masks <= 0 or n_activations <= 0:
        raise ValueError("counts must be positive")
    if not 0.0 <= keep_prob <= 1.0:
        raise ValueError("keep_prob must be between 0 and 1")

    rng = np.random.default_rng(seed)
    # Local contract: return integer `counts` with shape `(n_masks,)` and float summaries.
    # `empirical_mean` and population `empirical_variance` use sampled counts with `ddof=0`; theory describes the fixed model.
    # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
    # TODO: counts를 생성하고 관측 평균·분산과 이론 평균·분산을 계산하세요.
    counts = rng.binomial(n_activations, keep_prob, n_masks)
    empirical_mean = np.mean(counts)
    empirical_variance = np.var(counts)
    theoretical_mean = n_activations * keep_prob
    theoretical_variance = n_activations * keep_prob * (1 - keep_prob)
    # 학습자 편집 구간 끝
    return {
        "counts": counts,
        "empirical_mean": empirical_mean,
        "empirical_variance": empirical_variance,
        "theoretical_mean": theoretical_mean,
        "theoretical_variance": theoretical_variance,
    }

# 실행 순서: 이 셀 → 바로 아래 fixture → check_e02()

In [6]:
summary = sample_retained_counts(n_masks=8, n_activations=4, keep_prob=0.75, seed=11)
for name, value in summary.items():
    print(f"{name}: {value}")

counts: [4 3 3 4 4 2 4 4]
empirical_mean: 3.5
empirical_variance: 0.5
theoretical_mean: 3.0
theoretical_variance: 0.75


In [7]:
def check_e02() -> None:
    first = sample_retained_counts(8, 4, 0.75, 11)
    second = sample_retained_counts(8, 4, 0.75, 11)
    counts = first["counts"]
    np.testing.assert_equal(isinstance(counts, np.ndarray), True)
    np.testing.assert_equal(counts.shape, (8,))
    np.testing.assert_equal(np.issubdtype(counts.dtype, np.integer), True)
    np.testing.assert_array_equal(counts, second["counts"])
    np.testing.assert_allclose(first["empirical_mean"], np.mean(counts))
    np.testing.assert_allclose(first["empirical_variance"], np.var(counts))
    np.testing.assert_allclose(first["theoretical_mean"], 3.0)
    np.testing.assert_allclose(first["theoretical_variance"], 0.75)
    zero = sample_retained_counts(3, 1, 0.0, 9)
    np.testing.assert_array_equal(zero["counts"], np.zeros(3, dtype=int))
    one = sample_retained_counts(3, 1, 1.0, 9)
    np.testing.assert_array_equal(one["counts"], np.ones(3, dtype=int))
    for args in ((0, 4, 0.75, 1), (2, 0, 0.75, 1), (2, 4, -0.1, 1)):
        try:
            sample_retained_counts(*args)
        except ValueError:
            pass
        else:
            raise AssertionError(f"ValueError expected for {args}")
    print("check_e02 passed")

check_e02()

check_e02 passed


### 결과 해석

<!-- TODO: 관측 평균·분산이 이론값과 달라도 오류라고 단정할 수 없는 이유와, 구조화된 dropout에서 이 실습의 독립 가정이 왜 깨질 수 있는지를 3~5문장으로 설명하세요. -->

이번 실험의 관측 평균과 분산은 마스크를 8번만 뽑아 계산했으므로 이론값과 조금 다를 수 있다. 이는 유한한 표본에서 생기는 우연한 차이이므로 바로 오류라고 단정할 수 없다. 마스크를 훨씬 많이 생성하면 관측값은 이론값에 가까워질 것으로 예상된다. 구조화된 dropout에서는 채널이나 활성값 묶음이 함께 유지·제거되므로, 한 활성값의 결과가 다른 활성값과 독립이라는 가정이 깨진다.